# Extract relevant text using Regex
- Replace Patient Name with John Doe4
- Remove Lines beginning with "Booking Centre", "Test Performed at", "The authenticity"

This reduces the number of tokens that would be sent later.

Then save the result into another table **documents_clean**

### Prompt used

```
You are a Python data-cleaning assistant.

## Context
I have a SQLite database `reports.db` with a table `documents` containing
columns: id, filename, content. The `content` column holds multi-page text
extracted from medical PDF reports.

## Task
Read `reports.db` and produce a NEW database `reports_clean.db` (same schema).
Do NOT modify the source database.

Apply the following cleaning rules to the `content` column of every row:

### Rule 1 — Anonymise Patient Name
Replace the patient's name on every line matching the pattern:
  Patient Name : <NAME> Centre :
with:
  Patient Name : John Doe4 Centre :

The name sits between the literal tokens "Patient Name : " and " Centre :".

### Rule 2 — Remove Lines
Delete any line (in its entirety) that begins with one of these prefixes:
  - "Booking Centre"
  - "Test Performed at"
  - "The authenticity"

## Implementation
- Use Python's `re` module for both rules.
- Open `reports.db` in read-only mode (sqlite URI: `?mode=ro`).
- Write cleaned rows to table `documents_clean`.
- Print a per-row summary: filename, lines before → after, lines removed.

## Sample Input (one page of `content`)
--- Page 1 ---
Laboratory Investigation Report
Patient Name : Mr. Suumit Doe Centre : 5544 - Visit Health Pvt Ltd -Corporate
Age/Gender : 44 Y 10 M 10 D / M
Test Performed at :7091 - MAX LAB UDYOG VIHAR PHASE 5 D50, 2nd Floor
Booking Centre :5544 - Visit Health Pvt Ltd -Corporate, 237, Okhla
The authenticity of the report can be verified by scanning the Q R Code

## Expected Output (same page after cleaning)
--- Page 1 ---
Laboratory Investigation Report
Patient Name : John Doe4 Centre : 5544 - Visit Health Pvt Ltd -Corporate
Age/Gender : 44 Y 10 M 10 D / M
```

In [2]:
# Read
"""
clean_db_content.py
-------------------
Reads the 'documents' table from reports.db, then for each row:
  1. Replaces the Patient Name value with "John Doe4"
  2. Removes any line that begins with:
       - "Booking Centre"
       - "Test Performed at"
       - "The authenticity"

The cleaned content is written back to the same row (in-place update).
A separate table 'documents_cleaned' is also created as a safe copy.

Requirements: reports.db must exist (run pdf_to_sqlite.py first)
"""

import sqlite3
import re
from pathlib import Path

DB_PATH = "reports.db"

# ── Regex patterns ────────────────────────────────────────────────────────────

# Matches "Patient Name : Mr./Mrs. Any Name" and captures everything up to
# the next field separator (two or more spaces, or end of line).
# Works for names like "Mr. Suumit Doe", "Mrs. Jane Smith", "John Doe", etc.
PATIENT_NAME_RE = re.compile(
    r'(?<=Patient Name : )(.+?)(?= Centre :)',
    re.IGNORECASE
)

# Lines to drop: strip the line if it *starts* with any of these prefixes
# (after optional leading whitespace).
DROP_LINE_PREFIXES = re.compile(
    r'^\s*(Booking\s+Centre|Test\s+Performed\s+at|The\s+authenticity)',
    re.IGNORECASE | re.MULTILINE
)

REPLACEMENT_NAME = "John Doe4"


# ── Transformation functions ──────────────────────────────────────────────────

def replace_patient_name(text: str) -> str:
    """Replace the patient name value with REPLACEMENT_NAME."""
    return PATIENT_NAME_RE.sub(REPLACEMENT_NAME, text)


def remove_flagged_lines(text: str) -> str:
    """Remove entire lines that start with the flagged prefixes."""
    cleaned_lines = [
        line for line in text.splitlines()
        if not DROP_LINE_PREFIXES.match(line)
    ]
    return "\n".join(cleaned_lines)


def clean_content(text: str) -> str:
    """Apply all transformations to a content string."""
    text = replace_patient_name(text)
    text = remove_flagged_lines(text)
    return text


# ── Database helpers ──────────────────────────────────────────────────────────

def setup_cleaned_table(conn: sqlite3.Connection):
    """Create a fresh documents_cleaned table."""
    conn.execute("DROP TABLE IF EXISTS documents_cleaned")
    conn.execute("""
        CREATE TABLE documents_cleaned (
            id       INTEGER PRIMARY KEY AUTOINCREMENT,
            filename TEXT NOT NULL,
            content  TEXT
        )
    """)
    conn.commit()


def process_db(db_path: str):
    if not Path(db_path).exists():
        print(f"ERROR: '{db_path}' not found. Run pdf_to_sqlite.py first.")
        return

    conn = sqlite3.connect(db_path)
    setup_cleaned_table(conn)

    rows = conn.execute("SELECT id, filename, content FROM documents").fetchall()
    print(f"Found {len(rows)} record(s) in 'documents'.\n")

    for row_id, filename, content in rows:
        if not content:
            print(f"  [SKIP] id={row_id} '{filename}' — empty content.")
            continue

        cleaned = clean_content(content)

        # Write back to original table
        conn.execute(
            "UPDATE documents SET content = ? WHERE id = ?",
            (cleaned, row_id)
        )

        # Also insert into the safe copy table
        conn.execute(
            "INSERT INTO documents_cleaned (filename, content) VALUES (?, ?)",
            (filename, cleaned)
        )

        conn.commit()

        orig_lines  = len(content.splitlines())
        clean_lines = len(cleaned.splitlines())
        print(f"  [OK] id={row_id}  file='{filename}'")
        print(f"       Lines before: {orig_lines}  →  after: {clean_lines}  "
              f"(removed {orig_lines - clean_lines} line(s))")

    conn.close()
    print(f"\nDone. Both 'documents' and 'documents_cleaned' are updated in '{db_path}'.")



# ── Entry point ───────────────────────────────────────────────────────────────

if __name__ == "__main__":
    process_db(DB_PATH)

Found 2 record(s) in 'documents'.

  [OK] id=1  file='rep1.pdf'
       Lines before: 60  →  after: 60  (removed 0 line(s))
  [OK] id=2  file='rep2.pdf'
       Lines before: 59  →  after: 59  (removed 0 line(s))

Done. Both 'documents' and 'documents_cleaned' are updated in 'reports.db'.
